In [1]:
#格式化书本
from __future__ import division, print_function
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 交互式演示

这是本书中交互式演示的合集。如果你正在阅读本书的印刷版，或者通过 Github 或 nbviewer 在线阅读，你将无法运行这些交互。

因此我创建了这个 notebook。如果你的电脑上没有安装 IPython，可以按照以下步骤运行交互：

1. 在浏览器中访问 try.jupyter.org。它会为你启动一个临时的 notebook 服务器。

2. 点击 **New** 按钮并选择 `Python 3`。这将创建一个新的 notebook，在浏览器中运行 Python 3。

3. 将本 notebook 中某个 cell 的全部内容复制，粘贴到浏览器中 notebook 的 'code' cell 中。

4. 按 CTRL+ENTER 执行该 cell。

5. 尽情尝试吧！修改代码，随意实验。

你的服务器和 notebook 不会被永久保存。关闭会话后数据就会丢失。虽然点击保存时会显示保存成功，你也能在目录中看到文件，但这只是在 Docker 容器中进行的，窗口关闭后就会被删除。请将需要保留的更改复制粘贴到外部文件中。

当然，如果你安装了 IPython，可以下载这个 notebook 在自己的电脑上运行。在下载此文件的目录中，在命令提示符下输入

    ipython notebook
    
然后点击文件名即可打开。

# FPF' 实验

卡尔曼滤波器在预测步骤中使用方程 $P^- = FPF^\mathsf{T}$ 来计算协方差矩阵的先验，其中 P 是协方差矩阵，F 是系统状态转移函数。对于牛顿系统 $x = \dot{x}\Delta t + x_0$，F 的形式为

$$F = \begin{bmatrix}1 & \Delta t\\0 & 1\end{bmatrix}$$

$FPF^\mathsf{T}$ 通过考虑位置 ($x$) 和速度 ($\dot{x}$) 之间的相关性来改变 P。这个交互式图表让你看到 F 的不同设计对这个值的影响。例如，

* 如果 $x$ 和 $\dot{x}$ 不相关？（将 F01 设为 0）

* 如果 $x = 2\dot{x}\Delta t + x_0$？（将 F01 设为 2）

* 如果 $x = \dot{x}\Delta t + 2*x_0$？（将 F00 设为 2）

* 如果 $x = \dot{x}\Delta t$？（将 F00 设为 0）

In [2]:
%matplotlib inline
from IPython.html.widgets import interact, interactive, fixed
import IPython.html.widgets as widgets
import numpy as np
import numpy.linalg as linalg
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(x, P, edgecolor='k'):
    U,s,v = linalg.svd(P)
    angle = math.atan2(U[1,0],U[0,0])
    width  = math.sqrt(s[0]) * 2
    height = math.sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor=edgecolor, facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    ax.set_aspect('equal')
    
    
def plot_FPFT(F00, F01, F10, F11, covar):
    
    dt = 1.
    x = np.array((0, 0.))
    P = np.array(((1, covar), (covar, 2)))
    F = np.array(((F00, F01), (F10, F11)))

    plot_covariance_ellipse(x, P)
    plot_covariance_ellipse(x, np.dot(F, P).dot(F.T), edgecolor='r')
    #plt.axis('equal')
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.title(str(F))
    plt.xlabel('位置')
    plt.ylabel('速度')
                 
interact(plot_FPFT, 
         F00=widgets.IntSlider(value=1, min=0, max=2.), 
         F01=widgets.FloatSlider(value=1, min=0., max=2., description='F01(dt)'),
         F10=widgets.FloatSlider(value=0, min=0., max=2.),
         F11=widgets.FloatSlider(value=1, min=0., max=2.),
         covar=widgets.FloatSlider(value=0, min=0, max=1.));

C:\Anaconda3\lib\site-packages\IPython\html.py:14: ShimWarning: The `IPython.html` package has been deprecated since IPython 4.0. You should import from `notebook` instead. `IPython.html.widgets` has moved to `ipywidgets`.
  "`IPython.html.widgets` has moved to `ipywidgets`.", ShimWarning)


interactive(children=(IntSlider(value=1, description='F00', max=2), FloatSlider(value=1.0, description='F01(dt…

# 协方差椭圆

观察协方差矩阵中不同方差和协方差值的影响，矩阵形式为

$$\begin{bmatrix}\texttt{var}_x & \texttt{cov}_xy \\ \texttt{cov}_xy & \texttt{var}_y\end{bmatrix}$$

In [3]:
%matplotlib inline
from IPython.html.widgets import interact, interactive, fixed
from IPython.html.widgets import FloatSlider
from math import cos, sin, pi, atan2, sqrt
import  numpy.linalg as linalg
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(P):
    U,s,v = linalg.svd(P)
    angle = atan2(U[1,0],U[0,0])
    width  = sqrt(s[0]) * 2
    height = sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor='k', facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    h, w = height/4, width/4
    plt.plot([0, h*cos(angle+pi/2)], [0, h*sin(angle+pi/2)])
    plt.plot([0, w*cos(angle)],      [0, w*sin(angle)])

def plot_covariance(var_x, var_y, cov_xy):
    P = [[var_x, cov_xy], [cov_xy, var_y]]
    plot_covariance_ellipse(P)
    plt.xlim(-6, 6)
    plt.gca().set_aspect('equal')
    plt.ylim(-6, 6)
    plt.show()

interact (plot_covariance,           
          var_x=FloatSlider(value=5., min=0, max=20.), 
          var_y=FloatSlider(value=5., min=0., max=20.), 
          cov_xy=FloatSlider(value=1.5, min=0.0, max=50, step=.2));


interactive(children=(FloatSlider(value=5.0, description='var_x', max=20.0), FloatSlider(value=5.0, descriptio…

# g-h 滤波器

尝试 g-h 滤波器参数的不同值。

In [4]:
%matplotlib inline
from IPython.html.widgets import interact, interactive, fixed
import IPython.html.widgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import numpy.random as random

def gen_data(x0, dx, count, noise_factor):
    return [x0 + dx*i + random.randn()*noise_factor for i in range (count)]

def g_h_filter(data, x0, dx, g, h, dt=1., pred=None):    
    x = x0
    results = []
    for z in data:
        #prediction step
        x_est = x + (dx*dt)
        dx = dx        
        if pred is not None:
            pred.append(x_est)
        
        # update step
        residual = z - x_est
        dx = dx    + h * (residual) / dt
        x  = x_est + g * residual     
        results.append(x)  
    return np.array(results)

zs = gen_data(x0=5, dx=5, count=100, noise_factor=50)

def interactive_gh(x, dx, g, h):
    data = g_h_filter(data=zs, x0=x, dx=dx, dt=1.,g=g, h=h)
    plt.plot(zs, color='r')
    plt.plot(data, color='k')
    plt.show()

interact (interactive_gh,           
          x=widgets.FloatSlider(value=0., min=-50, max=50.), 
          dx=widgets.FloatSlider(value=5., min=-50., max=50.), 
          g=widgets.FloatSlider(value=0.1, min=0.01, max=2, step=.02), 
          h=widgets.FloatSlider(value=0.02, min=0.0, max=0.5, step=0.01));

interactive(children=(FloatSlider(value=0.0, description='x', max=50.0, min=-50.0), FloatSlider(value=5.0, des…